# 05 — Validación Económica y Selección Final

Tras seleccionar tres configuraciones finalistas (una por modelo de representación: TF-IDF, FinBERT y SBERT) basándonos en criterios geométricos, en este cuaderno evaluamos su utilidad financiera. El objetivo es comprobar si las agrupaciones generadas a partir del texto capturan mejor el comovimiento de los retornos bursátiles que la clasificación oficial GICS.

Para ello, utilizamos dos pruebas evaluadas mediante un análisis *Walk-Forward* y siempre comparando los resultados frente a los del GICS; nuestro benchmark.

1. **Prueba 1 (Coherencia de comovimiento):** Correlación de Spearman intra-grupo con retornos semanales.
2. **Prueba 2 (Modelo de Índice Único Sectorial):** Cálculo del error predictivo (RMSE) de los residuos al predecir los retornos usando un factor sintético del grupo.

En la decisión final priorizaremos la reducción del error RMSE frente a GICS y utilizaremos la correlación intra-clúster únicamente para desempatar.

## 1. Configuración, rutas y funciones matemáticas

Primero definimos las rutas de lectura y escritura. También agrupamos en esta sección las funciones base (transformación Fisher-Z, Spearman, cálculo de RMSE y emparejamiento temporal) para garantizar que todas las fases del código evalúen las métricas exactamente con la misma lógica.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import hdbscan
import statsmodels.api as sm
import umap.umap_ as umap
from scipy.sparse import issparse
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

Definimos los directorios de trabajo. Cargamos los datos procesados en los notebooks anteriores y preparamos la carpeta de salida para exportar los resultados.

In [2]:
PROJECT_ROOT = Path.cwd()

PATHS = {
    "dataset_lematizado": PROJECT_ROOT / "data" / "processed" / "Dataset_lematizado.parquet",
    "log_retornos":       PROJECT_ROOT / "data" / "raw" / "Log_Retornos_2019_2024.parquet",
    "sectores_gics_raw":  PROJECT_ROOT / "data" / "raw" / "Sectores_GICS_raw.parquet",
    "finalistas_fase1":   PROJECT_ROOT / "experiments" / "Seleccion_Finalistas.csv",
    "matriz_finbert":     PROJECT_ROOT / "data" / "matrices" / "Matriz_FinBERT.npy",
    "matriz_sbert":       PROJECT_ROOT / "data" / "matrices" / "Matriz_SBERT.npy",
    "clusters_out":       PROJECT_ROOT / "outputs",
}
PATHS["clusters_out"].mkdir(parents=True, exist_ok=True)

PARAMS_TFIDF = {"max_df": 0.85, "min_df": 0.01, "max_features": 5000}

UMAP_PARAMS = {
    "n_neighbors": 15,
    "n_components": 50,
    "metric": "cosine",
    "random_state": RANDOM_STATE,
}

WALK_FORWARD = {"train_min_end_year": 2020}

**Promedio de Fisher-Z para correlaciones.** Como hacer una media aritmética directa de coeficientes acotados entre $[-1, 1]$ genera sesgos, utilizamos la transformación $z_i = \operatorname{arctanh}(\rho_i)$ para promediar en un espacio no acotado y luego revertimos el resultado. Aplicamos un recorte en $\pm 0.9999$ para evitar que la transformación genere valores infinitos en el caso de que existan correlaciones perfectas.

In [3]:
def fisher_z_mean(valores, clip=0.9999):
    v = np.asarray(valores, dtype=float)
    v = v[~np.isnan(v)]
    if v.size == 0:
        return float("nan")
    v = np.clip(v, -clip, clip)
    return float(np.tanh(np.mean(np.arctanh(v))))

**Matriz de Spearman.** Calculamos la correlación de Pearson directamente sobre el rango de los retornos semanales. Para resumir la cohesión interna de un grupo en un solo número, hacemos la media Fisher-Z del triángulo superior de la matriz de correlación (por ser una matriz simétrica).

In [4]:
def matriz_spearman(ret_subwindow):
    return ret_subwindow.rank(method="average").corr(method="pearson")


def correlacion_grupo(tickers_grupo, matriz_corr):
    tickers = [t for t in tickers_grupo if t in matriz_corr.index]
    if len(tickers) < 2:
        return np.nan
    sub = matriz_corr.loc[tickers, tickers]
    return fisher_z_mean(sub.values[np.triu_indices_from(sub, k=1)])


def spearman_intra(df_grupos, col_grupo, matriz_corr):
    rhos = [
        correlacion_grupo(
            df_grupos.loc[df_grupos[col_grupo] == g, "ticker"].unique(),
            matriz_corr,
        )
        for g in df_grupos[col_grupo].dropna().unique()
    ]
    rhos = [r for r in rhos if not np.isnan(r)]
    return fisher_z_mean(np.array(rhos)) if rhos else np.nan


**Modelo de Índice Único Sectorial.** Para cada empresa, creamos un factor sintético calculando la media de los retornos de sus compañeras de grupo. Luego, ajustamos una regresión lineal y extraemos la suma de los residuos al cuadrado y el número de observaciones reales. Lo hacemos así en lugar de calcular directamente el RMSE por trimestre para evitar sesgos al agregar y promediar los datos a nivel anual.

In [5]:
def rmse_indice_unico(ticker_grupo_dict, ret_subwindow):
    grupos = {}
    for t, g in ticker_grupo_dict.items():
        grupos.setdefault(g, []).append(t)

    sum_sq_res_list, n_obs_list = [], []
    for tickers_g in grupos.values():
        tickers_validos = [t for t in tickers_g if t in ret_subwindow.columns]
        if len(tickers_validos) < 2:
            continue
        for ticker in tickers_validos:
            otros = [t for t in tickers_validos if t != ticker]
            if not otros:
                continue
            F_g = ret_subwindow[otros].mean(axis=1)
            r_i = ret_subwindow[ticker]
            valid = F_g.notna() & r_i.notna()
            if valid.sum() < 5:
                continue
            X = sm.add_constant(F_g[valid].values)
            y = r_i[valid].values
            modelo = sm.OLS(y, X).fit()
            sum_sq_res_list.append(float(np.sum(modelo.resid ** 2)))
            n_obs_list.append(int(valid.sum()))

    return float(np.sum(sum_sq_res_list)), int(np.sum(n_obs_list))

### Detalle por regresión

La función rmse_indice_unico agrega todos los residuos en un único RMSE global, lo que basta para el cálculo de la reducción de RMESE frente al GICS de cada modelo, pero no permite cuantificar la incertidumbre de la diferencia frente a GICS. Para ello, se añade una variante que conserva el detalle de cada regresión empresa-trimestre: para cada empresa k se almacena su suma de residuos al cuadrado (SSR_k) y su número de observaciones (n_k). El cálculo del RMSE agregado no se altera (el filtro de datos y la construcción del factor sintético son idénticos). Sobre este desglose se construirá en un bootstrap por bloques de ticker, que respeta la dependencia entre las observaciones de una misma empresa al remuestrear tickers completos en lugar de observaciones individuales.

In [6]:
def rmse_indice_unico_detalle(ticker_grupo_dict, ret_subwindow):
    """Variante de rmse_indice_unico que NO agrega: devuelve, por cada regresion
    empresa-trimestre valida, una tupla (ticker, suma_residuos_cuadrado, n_obs).
    Reutilizada por el bootstrap por bloques. La logica de
    construccion del factor y el filtro (>=2 tickers, >=5 obs) es identica a la
    de rmse_indice_unico, de modo que el RMSE agregado coincide exactamente."""
    grupos = {}
    for t, g in ticker_grupo_dict.items():
        grupos.setdefault(g, []).append(t)

    detalle = []  # (ticker, ssr, n)
    for tickers_g in grupos.values():
        tickers_validos = [t for t in tickers_g if t in ret_subwindow.columns]
        if len(tickers_validos) < 2:
            continue
        for ticker in tickers_validos:
            otros = [t for t in tickers_validos if t != ticker]
            if not otros:
                continue
            F_g = ret_subwindow[otros].mean(axis=1)
            r_i = ret_subwindow[ticker]
            valid = F_g.notna() & r_i.notna()
            if valid.sum() < 5:
                continue
            X = sm.add_constant(F_g[valid].values)
            y = r_i[valid].values
            modelo = sm.OLS(y, X).fit()
            detalle.append((ticker, float(np.sum(modelo.resid ** 2)), int(valid.sum())))
    return detalle


**Creación del espacio vectorial por partición (fold).** Para evitar sesgos de anticipación, en TF-IDF el vocabulario se entrena exclusivamente con datos del pasado ($\le T$). En FinBERT y SBERT, al ser embeddings contextuales preentrenados, simplemente filtramos la matriz global. Después, aplicamos UMAP y HDBSCAN sobre este conjunto histórico para proyectar y agrupar.

In [7]:
def vectores_anio(df_train, repr_name, matriz_global, idx_train):
    if repr_name == "tfidf":
        textos = df_train["presentation_limpia"].astype(str).tolist()
        vectorizer = TfidfVectorizer(
            max_df=PARAMS_TFIDF["max_df"],
            min_df=PARAMS_TFIDF["min_df"],
            max_features=PARAMS_TFIDF["max_features"],
            ngram_range=(1, 2),
            sublinear_tf=True,
        )
        mat = vectorizer.fit_transform(textos)
        return mat.toarray() if issparse(mat) else mat
    return matriz_global[idx_train]


def proyectar_y_clusterizar(matriz_local, params_hdbscan):
    reducer = umap.UMAP(
        n_neighbors=UMAP_PARAMS["n_neighbors"],
        n_components=UMAP_PARAMS["n_components"],
        metric=UMAP_PARAMS["metric"],
        random_state=UMAP_PARAMS["random_state"],
    )
    embeddings = reducer.fit_transform(matriz_local)
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=params_hdbscan["min_cluster_size"],
        min_samples=params_hdbscan["min_samples"],
        cluster_selection_epsilon=params_hdbscan["cluster_selection_epsilon"],
        cluster_selection_method=params_hdbscan["cluster_selection_method"],
        metric="euclidean",
    )
    labels = clusterer.fit_predict(embeddings)
    return embeddings, labels

**Mapeo de etiquetas.** Asignamos a cada empresa el clúster de su última earnings call* publicada antes de evaluar el trimestre. Se filtra por la fecha de publicación real (`earnings_date`) y no por el trimestre fiscal reportado, para garantizar que el modelo predictivo solo utiliza información que ya era pública en la fecha de análisis.

In [8]:
def ticker_label_map(df_year, labels_year, fecha_corte=None):
    columnas = ["ticker", "quarter"]
    if fecha_corte is not None:
        if "earnings_date" not in df_year.columns:
            raise ValueError(
                "ticker_label_map: falta 'earnings_date' pero se pasó fecha_corte."
            )
        columnas = ["ticker", "quarter", "earnings_date"]

    df = df_year[columnas].copy()
    df["label"] = labels_year
    df = df[df["label"] != -1]
    if df.empty:
        return {}

    if fecha_corte is not None:
        df = df[df["earnings_date"] < pd.Timestamp(fecha_corte)]
        if df.empty:
            return {}
        return (
            df.sort_values("earnings_date")
              .groupby("ticker").tail(1)
              .set_index("ticker")["label"].to_dict()
        )

    return (
        df.sort_values("quarter")
          .groupby("ticker").tail(1)
          .set_index("ticker")["label"].to_dict()
    )

**Ventanas trimestrales.** Dividimos el año de evaluación en cuatro trimestres. Solo consideramos grupos que tengan al menos 2 empresas válidas y exigimos un mínimo de 5 semanas de retornos reales para asegurar que las correlaciones y las regresiones tengan suficiente potencia estadística.

In [9]:
def buckets_trimestrales(year_eval):
    quarters = []
    for q in range(1, 5):
        start_month = (q - 1) * 3 + 1
        fecha_ini = pd.Timestamp(year=year_eval, month=start_month, day=1)
        if start_month + 3 > 12:
            fecha_fin = pd.Timestamp(year=year_eval, month=12, day=31)
        else:
            fecha_fin = (
                pd.Timestamp(year=year_eval, month=start_month + 3, day=1)
                - pd.Timedelta(days=1)
            )
        quarters.append((fecha_ini, fecha_fin))
    return quarters


def retornos_trimestre(df_retornos, fecha_ini, fecha_fin, tickers_supervivientes):
    mask = (df_retornos.index >= fecha_ini) & (df_retornos.index <= fecha_fin)
    fechas = df_retornos.index[mask]
    if len(fechas) < 5:
        return None
    cols = [t for t in tickers_supervivientes if t in df_retornos.columns]
    if len(cols) < 2:
        return None
    sub = df_retornos.loc[fechas, cols]
    sub = sub.loc[:, sub.notna().sum(axis=0) >= 5]
    return sub if sub.shape[1] >= 2 else None

## 2. Carga de datos y modelo de referencia (GICS)

Cargamos el texto procesado, los log-retornos semanales y las etiquetas GICS. Para el GICS, verificamos a qué sector pertenecía cada empresa en ese trimestre específico, reflejando así cualquier cambio real en la clasificación.

In [10]:
df_nlp = pd.read_parquet(PATHS["dataset_lematizado"])

df_retornos = pd.read_parquet(PATHS["log_retornos"])
# El parquet de log-retornos guarda las semanas en el indice. Si al releer
# vinieran como columna, la reponemos como indice antes de nada.
if not isinstance(df_retornos.index, pd.DatetimeIndex):
    col_fecha = next((c for c in df_retornos.columns
                      if str(c).lower() in ("date", "index", "fecha")), None)
    if col_fecha is not None:
        df_retornos = df_retornos.set_index(col_fecha)
df_retornos.index = pd.to_datetime(df_retornos.index)

df_gics_map = pd.read_parquet(PATHS["sectores_gics_raw"])
df_gics_map["quarter"] = df_gics_map["quarter"].astype(str)
df_gics_map = (
    df_gics_map[["ticker", "quarter", "gics_sector"]]
    .drop_duplicates(subset=["ticker", "quarter"], keep="first")
)

print(f"Documentos NLP:        {len(df_nlp):,}")
print(f"Tickers en retornos:   {df_retornos.shape[1]:,}")
print(f"Semanas en retornos:   {df_retornos.shape[0]:,}")
print(f"Filas panel GICS:      {len(df_gics_map):,}")
print(f"Rango retornos:        {df_retornos.index.min().date()} -> {df_retornos.index.max().date()}")

Documentos NLP:        8,893
Tickers en retornos:   482
Semanas en retornos:   273
Filas panel GICS:      10,563
Rango retornos:        2019-01-11 -> 2024-03-29


Cargamos la tabla con las tres configuraciones finalistas que seleccionamos en el cuaderno anterior, recuperando sus hiperparámetros óptimos de UMAP y HDBSCAN.

In [11]:
df_finalistas = pd.read_csv(PATHS["finalistas_fase1"])

# El experiment_id del notebook 04 identifica de forma unica cada candidato;
# lo usamos como nombre del finalista en lo que sigue.
finalistas = []
for i, row in df_finalistas.iterrows():
    exp_id = row["experiment_id"]
    finalistas.append({
        "nombre":         exp_id,
        "nombre_corto":   exp_id,
        "representation": row["representation"],
        "params": {
            "min_cluster_size":          int(row["min_cluster_size"]),
            "min_samples":               int(row["min_samples"]),
            "cluster_selection_epsilon": float(row["cluster_selection_epsilon"]),
            "cluster_selection_method":  row["cluster_selection_method"],
        },
        "rank": int(row.get("rank", i + 1)),
    })

for fin in finalistas:
    print(f"  #{fin['rank']}: {fin['nombre_corto']} [{fin['representation']}]")

  #1: finbert_mcs400_ms15_eps0.0_leaf [finbert]
  #2: tfidf_mcs150_ms30_eps0.0_eom [tfidf]
  #3: sbert_mcs150_ms30_eps0.0_leaf [sbert]


Cargamos en memoria las matrices densas de FinBERT y SBERT para optimizar el tiempo de ejecución. En el caso de TF-IDF, la matriz se recalculará dinámicamente en cada partición temporal.

In [12]:
def cargar_matriz_global(repr_name):
    if repr_name == "tfidf":
        return None
    clave = {"finbert": "matriz_finbert", "sbert": "matriz_sbert"}[repr_name]
    matriz = np.load(PATHS[clave])
    if matriz.shape[0] != len(df_nlp):
        raise AssertionError(
            f"Desalineación de cardinalidad en {repr_name!r}: "
            f"matriz {matriz.shape[0]} filas vs dataset {len(df_nlp)}."
        )
    return matriz


matriz_cache = {}
for fin in finalistas:
    r = fin["representation"]
    if r not in matriz_cache:
        matriz_cache[r] = cargar_matriz_global(r)
print("Representaciones cacheadas:", list(matriz_cache.keys()))

Representaciones cacheadas: ['finbert', 'tfidf', 'sbert']


**Definición de las ventanas predictivas.** Establecemos los años de entrenamiento ($T$) y de evaluación ($T+1$), asegurando que las pruebas operen exclusivamente sobre los periodos que disponen de datos de retornos futuros.

In [13]:
anios_disponibles = sorted({int(str(q)[:4]) for q in df_nlp["quarter"].astype(str).unique()})
anios_retornos = sorted({d.year for d in df_retornos.index})

anios_T = [
    y for y in anios_disponibles
    if y >= WALK_FORWARD["train_min_end_year"] and (y + 1) in anios_retornos
]
print(f"Años T evaluados (T -> T+1): {anios_T}")

Años T evaluados (T -> T+1): [2020, 2021, 2022, 2023]


## 3. Prueba 1: Coherencia de comovimiento (Spearman)

Evaluamos la correlación interna de cada grupo aplicando un enfoque predictivo (*Walk-Forward*). 

Para ser conservadores en el estudio, permitimos que el benchmark GICS utilice información contemporánea (evaluando los sectores reales del año $T+1$), mientras que HDBSCAN solo agrupa utilizando textos estrictamente del pasado ($\le T$). Solo hay un único cambio en la ventana temporal escogida, pero en cualquier caso existe una pequeña desventaja de nuestro modelo, por lo que los resultados pueden demostrar mayor solidez.
Además, cruzamos las bases de datos para garantizar que ambas metodologías se evalúen exactamente sobre el mismo universo de empresas.

In [14]:
def evaluar_candidato_spearman(finalista):
    repr_name = finalista["representation"]
    params = finalista["params"]
    matriz_global = matriz_cache[repr_name]
    registros_anuales = []

    # Opcion A: agregacion plana por trimestre. Acumulamos el rho de CADA
    # trimestre fuera de muestra en una unica bolsa global; el rho final del
    # modelo es la media de Fisher-Z sobre todos esos trimestres. Asi cada
    # trimestre pesa igual y un anio representado por un solo trimestre
    # (2024 -> Q1) NO cuenta como un anio completo. La equiponderacion ENTRE
    # clusteres dentro de cada trimestre (spearman_intra) se mantiene intacta.
    rho_hdb_q_all, rho_gics_q_all = [], []

    for year_T in anios_T:
        year_eval = year_T + 1

        anios_str = df_nlp["quarter"].astype(str).str[:4]
        mask_train = anios_str.astype(int) <= year_T
        idx_train = np.flatnonzero(mask_train.to_numpy())
        if idx_train.size < params["min_cluster_size"]:
            continue

        df_train = df_nlp.iloc[idx_train].reset_index(drop=True)
        mat_train = vectores_anio(df_train, repr_name, matriz_global, idx_train)

        _, labels_train = proyectar_y_clusterizar(mat_train, params)
        df_train["label"] = labels_train

        df_T = df_train[df_train["quarter"].astype(str).str.startswith(str(year_T))].copy()
        if df_T.empty:
            continue

        rho_hdb_q_list, rho_gics_q_list = [], []
        for fecha_ini, fecha_fin in buckets_trimestrales(year_eval):
            tk_label = ticker_label_map(df_T, df_T["label"].values, fecha_corte=fecha_ini)
            if not tk_label:
                continue

            target_quarter = f"{year_eval}Q{((fecha_ini.month - 1) // 3) + 1}"
            gics_eval_map = (
                df_gics_map[df_gics_map["quarter"] == target_quarter]
                .set_index("ticker")["gics_sector"]
            )

            universo_pareado = [
                t for t in tk_label.keys()
                if t in gics_eval_map.index and pd.notna(gics_eval_map[t])
            ]
            if len(universo_pareado) < 2:
                continue
            tk_label_int = {t: tk_label[t] for t in universo_pareado}
            tk_gics_int = {t: gics_eval_map[t] for t in universo_pareado}

            ret_q = retornos_trimestre(df_retornos, fecha_ini, fecha_fin, universo_pareado)
            if ret_q is None:
                continue
            matriz_corr_q = matriz_spearman(ret_q)

            tickers_finales = [t for t in universo_pareado if t in matriz_corr_q.index]
            if len(tickers_finales) < 2:
                continue

            df_grupos = pd.DataFrame({
                "ticker":  tickers_finales,
                "hdbscan": [tk_label_int[t] for t in tickers_finales],
                "sector":  [tk_gics_int[t] for t in tickers_finales],
            })

            rho_hdb_q = spearman_intra(df_grupos, "hdbscan", matriz_corr_q)
            rho_gics_q = spearman_intra(df_grupos, "sector", matriz_corr_q)
            if pd.notna(rho_hdb_q):
                rho_hdb_q_list.append(rho_hdb_q)
                rho_hdb_q_all.append(rho_hdb_q)        # pool global por trimestre
            if pd.notna(rho_gics_q):
                rho_gics_q_list.append(rho_gics_q)
                rho_gics_q_all.append(rho_gics_q)       # pool global por trimestre

        # La tabla anual se conserva SOLO como desglose descriptivo; ya NO define
        # el rho final del modelo, que ahora se promedia por trimestre.
        if rho_hdb_q_list or rho_gics_q_list:
            registros_anuales.append({
                "year_T": year_T,
                "year_eval": year_eval,
                "n_trimestres_hdb": len(rho_hdb_q_list),
                "n_trimestres_gics": len(rho_gics_q_list),
                "rho_hdbscan": fisher_z_mean(np.array(rho_hdb_q_list)) if rho_hdb_q_list else np.nan,
                "rho_gics": fisher_z_mean(np.array(rho_gics_q_list)) if rho_gics_q_list else np.nan,
            })

    if not registros_anuales:
        return None
    df_anual = pd.DataFrame(registros_anuales)
    return {
        "anual": df_anual,
        # Media de Fisher-Z sobre TODOS los trimestres OOS (ponderacion por trimestre).
        "rho_hdbscan": fisher_z_mean(np.array(rho_hdb_q_all)),
        "rho_gics": fisher_z_mean(np.array(rho_gics_q_all)),
        "n_anios_validos": int(df_anual["rho_hdbscan"].notna().sum()),
        "n_trimestres_validos": int(len(rho_hdb_q_all)),
    }

Ejecutamos la evaluación para los tres finalistas. Todos los promedios de correlación (intra-anuales e inter-anuales) se realizan utilizando el espacio Fisher-Z.

In [15]:
resultados_h1, detalle_h1 = [], []

for fin in finalistas:
    out = evaluar_candidato_spearman(fin)
    if out is None:
        resultados_h1.append({
            "nombre_corto": fin["nombre_corto"],
            "rho_hdbscan": np.nan, "rho_gics": np.nan,
            "delta_hdb_vs_gics": np.nan, "n_anios_validos": 0,
        })
        continue

    df_anual = out["anual"].copy()
    df_anual.insert(0, "nombre_corto", fin["nombre_corto"])
    detalle_h1.append(df_anual)

    resultados_h1.append({
        "nombre_corto": fin["nombre_corto"],
        "rho_hdbscan": round(out["rho_hdbscan"], 4),
        "rho_gics": round(out["rho_gics"], 4),
        "delta_hdb_vs_gics": round(out["rho_hdbscan"] - out["rho_gics"], 4),
        "n_anios_validos": out["n_anios_validos"],
    })
    print(f"  {fin['nombre_corto']:<28} rho_hdb={out['rho_hdbscan']:+.4f} | rho_gics={out['rho_gics']:+.4f}")

df_res_h1 = pd.DataFrame(resultados_h1)
df_res_h1.to_csv(PATHS["clusters_out"] / "Resultados_H1_Spearman_WF.csv", index=False)
if detalle_h1:
    pd.concat(detalle_h1, ignore_index=True).to_csv(
        PATHS["clusters_out"] / "Resultados_H1_Spearman_WF_anual.csv", index=False
    )
df_res_h1

  finbert_mcs400_ms15_eps0.0_leaf rho_hdb=+0.3776 | rho_gics=+0.4790
  tfidf_mcs150_ms30_eps0.0_eom rho_hdb=+0.5069 | rho_gics=+0.4859
  sbert_mcs150_ms30_eps0.0_leaf rho_hdb=+0.4972 | rho_gics=+0.5004


,nombre_corto,rho_hdbscan,rho_gics,delta_hdb_vs_gics,n_anios_validos
0,finbert_mcs400_ms15_eps0.0_leaf,0.3776,0.4790,-0.1014,4
1,tfidf_mcs150_ms30_eps0.0_eom,0.5069,0.4859,0.0210,4
2,sbert_mcs150_ms30_eps0.0_leaf,0.4972,0.5004,-0.0032,4


## 4. Prueba 2: Modelo de Índice Único Sectorial (RMSE)

Esta segunda prueba mide la capacidad de cada agrupación para explicar el comportamiento bursátil de sus integrantes. Usamos el mismo diseño predictivo y universo de empresas que en la prueba anterior.

Para no sesgar el cálculo final del RMSE, acumulamos los residuos al cuadrado y el número de observaciones a nivel anual antes de hacer la raíz cuadrada final. Al evaluar el error de predicción a través de los residuos puros de una regresión clásica, no es necesario aplicar correcciones de varianza, ya que estas no alteran los residuos del modelo.

In [16]:
def evaluar_candidato_indice_unico(finalista):
    repr_name = finalista["representation"]
    params = finalista["params"]
    matriz_global = matriz_cache[repr_name]
    registros = []

    for year_T in anios_T:
        year_eval = year_T + 1

        anios_str = df_nlp["quarter"].astype(str).str[:4]
        mask_train = anios_str.astype(int) <= year_T
        idx_train = np.flatnonzero(mask_train.to_numpy())
        if idx_train.size < params["min_cluster_size"]:
            continue

        df_train = df_nlp.iloc[idx_train].reset_index(drop=True)
        mat_train = vectores_anio(df_train, repr_name, matriz_global, idx_train)

        _, labels_train = proyectar_y_clusterizar(mat_train, params)
        df_train["label"] = labels_train

        df_T = df_train[df_train["quarter"].astype(str).str.startswith(str(year_T))].copy()
        if df_T.empty:
            continue

        total_ssr_hdb, total_n_hdb = 0.0, 0
        total_ssr_gics, total_n_gics = 0.0, 0

        for fecha_ini, fecha_fin in buckets_trimestrales(year_eval):
            tk_label = ticker_label_map(df_T, df_T["label"].values, fecha_corte=fecha_ini)
            if not tk_label:
                continue

            target_quarter = f"{year_eval}Q{((fecha_ini.month - 1) // 3) + 1}"
            gics_eval_map = (
                df_gics_map[df_gics_map["quarter"] == target_quarter]
                .set_index("ticker")["gics_sector"]
            )

            universo_pareado = [
                t for t in tk_label.keys()
                if t in gics_eval_map.index and pd.notna(gics_eval_map[t])
            ]
            if len(universo_pareado) < 2:
                continue
            tk_label_int = {t: tk_label[t] for t in universo_pareado}
            tk_gics_int = {t: gics_eval_map[t] for t in universo_pareado}

            ret_q = retornos_trimestre(df_retornos, fecha_ini, fecha_fin, universo_pareado)
            if ret_q is None:
                continue

            tickers_finales = [t for t in universo_pareado if t in ret_q.columns]
            if len(tickers_finales) < 2:
                continue
            if len(tickers_finales) < len(universo_pareado):
                tk_label_int = {t: tk_label_int[t] for t in tickers_finales}
                tk_gics_int = {t: tk_gics_int[t] for t in tickers_finales}

            ssr_hdb, n_hdb = rmse_indice_unico(tk_label_int, ret_q)
            ssr_gics, n_gics = rmse_indice_unico(tk_gics_int, ret_q)
            total_ssr_hdb += ssr_hdb
            total_n_hdb += n_hdb
            total_ssr_gics += ssr_gics
            total_n_gics += n_gics

        rmse_year_hdb = np.sqrt(total_ssr_hdb / total_n_hdb) if total_n_hdb > 0 else np.nan
        rmse_year_gics = np.sqrt(total_ssr_gics / total_n_gics) if total_n_gics > 0 else np.nan
        if pd.isna(rmse_year_hdb) or pd.isna(rmse_year_gics):
            continue

        registros.append({
            "year_T": year_T,
            "year_eval": year_eval,
            "rmse_hdbscan": float(rmse_year_hdb),
            "rmse_gics": float(rmse_year_gics),
            "ssr_hdb": float(total_ssr_hdb),
            "n_obs_hdb": int(total_n_hdb),
            "ssr_gics": float(total_ssr_gics),
            "n_obs_gics": int(total_n_gics),
        })

    if not registros:
        return None
    df_anual = pd.DataFrame(registros)

    sum_ssr_hdb = float(df_anual["ssr_hdb"].sum())
    sum_n_hdb = int(df_anual["n_obs_hdb"].sum())
    sum_ssr_gics = float(df_anual["ssr_gics"].sum())
    sum_n_gics = int(df_anual["n_obs_gics"].sum())

    rmse_pooled_hdb = np.sqrt(sum_ssr_hdb / sum_n_hdb) if sum_n_hdb > 0 else float("nan")
    rmse_pooled_gics = np.sqrt(sum_ssr_gics / sum_n_gics) if sum_n_gics > 0 else float("nan")

    return {
        "anual": df_anual,
        "rmse_hdbscan": float(rmse_pooled_hdb),
        "rmse_gics": float(rmse_pooled_gics),
        "n_anios_validos": int(df_anual["rmse_hdbscan"].notna().sum()),
    }

Calculamos el RMSE agregado de los tres finalistas y lo comparamos con el resultado de GICS. Un porcentaje de reducción negativo significa que nuestra agrupación textual logra un error de predicción menor (es decir, el modelo textual explica mejor los retornos que el estándar GICS).

In [17]:
resultados_iu, detalle_iu = [], []

for fin in finalistas:
    out = evaluar_candidato_indice_unico(fin)
    if out is None:
        resultados_iu.append({
            "nombre_corto": fin["nombre_corto"],
            "rmse_hdbscan": np.nan, "rmse_gics": np.nan,
            "reduccion_pct": np.nan, "n_anios_validos": 0,
        })
        continue

    df_anual = out["anual"].copy()
    df_anual.insert(0, "nombre_corto", fin["nombre_corto"])
    detalle_iu.append(df_anual)

    if pd.notna(out["rmse_gics"]) and out["rmse_gics"] > 1e-12 and pd.notna(out["rmse_hdbscan"]):
        reduccion_pct = (out["rmse_hdbscan"] - out["rmse_gics"]) / out["rmse_gics"] * 100
    else:
        reduccion_pct = float("nan")

    resultados_iu.append({
        "nombre_corto": fin["nombre_corto"],
        "rmse_hdbscan": round(out["rmse_hdbscan"], 4),
        "rmse_gics": round(out["rmse_gics"], 4),
        "reduccion_pct": round(reduccion_pct, 2) if pd.notna(reduccion_pct) else float("nan"),
        "n_anios_validos": out["n_anios_validos"],
    })
    msg = f"{reduccion_pct:+.2f}%" if pd.notna(reduccion_pct) else "n/d"
    print(f"  {fin['nombre_corto']:<28} rmse_hdb={out['rmse_hdbscan']:.4f} | rmse_gics={out['rmse_gics']:.4f} | red={msg}")

df_res_iu = pd.DataFrame(resultados_iu)
df_res_iu.to_csv(PATHS["clusters_out"] / "Resultados_IndiceUnico_WF.csv", index=False)
if detalle_iu:
    pd.concat(detalle_iu, ignore_index=True).to_csv(
        PATHS["clusters_out"] / "Resultados_IndiceUnico_WF_anual.csv", index=False
    )
df_res_iu

  finbert_mcs400_ms15_eps0.0_leaf rmse_hdb=0.0313 | rmse_gics=0.0290 | red=+8.16%
  tfidf_mcs150_ms30_eps0.0_eom rmse_hdb=0.0286 | rmse_gics=0.0290 | red=-1.46%
  sbert_mcs150_ms30_eps0.0_leaf rmse_hdb=0.0277 | rmse_gics=0.0277 | red=+0.06%


,nombre_corto,rmse_hdbscan,rmse_gics,reduccion_pct,n_anios_validos
0,finbert_mcs400_ms15_eps0.0_leaf,0.0313,0.0290,8.16,4
1,tfidf_mcs150_ms30_eps0.0_eom,0.0286,0.0290,-1.46,4
2,sbert_mcs150_ms30_eps0.0_leaf,0.0277,0.0277,0.06,4


## 5. Selección Final

Consolidamos los resultados en un ranking. La métrica principal para decidir el ganador es la reducción del error predictivo (RMSE) frente a GICS. Solo usaremos la correlación de Spearman en caso de empate.

In [18]:
df_ranking = pd.DataFrame([{
    "nombre_corto": fin["nombre_corto"],
    "nombre": fin["nombre"],
    "representation": fin["representation"],
    "rank_fase_1": fin["rank"],
    "rmse_hdbscan": m_iu["rmse_hdbscan"],
    "rmse_gics": m_iu["rmse_gics"],
    "reduccion_pct": m_iu["reduccion_pct"],
    "rho_hdbscan": m_h1["rho_hdbscan"],
    "rho_gics": m_h1["rho_gics"],
    "delta_rho": m_h1["delta_hdb_vs_gics"],
}
    for fin in finalistas
    for m_iu in [next(r for r in resultados_iu if r["nombre_corto"] == fin["nombre_corto"])]
    for m_h1 in [next(r for r in resultados_h1 if r["nombre_corto"] == fin["nombre_corto"])]
])

df_ranking["_sort_red"] = df_ranking["reduccion_pct"].fillna(np.inf)
df_ranking["_sort_rho_neg"] = -df_ranking["rho_hdbscan"].fillna(-np.inf)
df_ranking = (
    df_ranking.sort_values(by=["_sort_red", "_sort_rho_neg"], ascending=[True, True])
    .drop(columns=["_sort_red", "_sort_rho_neg"])
    .reset_index(drop=True)
)
df_ranking["rank_final"] = range(1, len(df_ranking) + 1)

cols_display = [
    "rank_final", "nombre_corto", "representation", "rank_fase_1",
    "reduccion_pct", "rmse_hdbscan", "rmse_gics", "rho_hdbscan", "rho_gics", "delta_rho",
]
df_ranking[cols_display]

,rank_final,nombre_corto,representation,rank_fase_1,reduccion_pct,rmse_hdbscan,rmse_gics,rho_hdbscan,rho_gics,delta_rho
0,1,tfidf_mcs150_ms30_eps0.0_eom,tfidf,2,-1.46,0.0286,0.0290,0.5069,0.4859,0.0210
1,2,sbert_mcs150_ms30_eps0.0_leaf,sbert,3,0.06,0.0277,0.0277,0.4972,0.5004,-0.0032
2,3,finbert_mcs400_ms15_eps0.0_leaf,finbert,1,8.16,0.0313,0.0290,0.3776,0.4790,-0.1014


**Comparación de la Geometría vs Desempeño Económico.** Contrastamos si la configuración que presentaba los clústeres matemáticamente más compactos (el mejor modelo de la Fase 1) es la misma que mejor predice los retornos bursátiles. Si las posiciones difieren, confirmaremos que una buena separación topológica en UMAP no se traduce automáticamente en valor y utilidad financiera.

In [19]:
fase1_lider = df_ranking[df_ranking["rank_fase_1"] == 1]
ganador = df_ranking.iloc[0]

if not fase1_lider.empty:
    rank_eco_lider1 = int(fase1_lider.iloc[0]["rank_final"])
    if rank_eco_lider1 > 1:
        print("DIVERGENCIA geométrico vs económico:")
        print(f"  Líder de Fase 1 cae a la posición económica #{rank_eco_lider1}.")
        print(f"  Ganador económico ocupaba la posición #{int(ganador['rank_fase_1'])} en Fase 1.")
    else:
        print("Convergencia: el líder geométrico es también el ganador económico.")

print("\nMODELO ÓPTIMO:", ganador["nombre"])
print("  Representación:    ", ganador["representation"].upper())
print(f"  Reducción vs GICS:  {ganador['reduccion_pct']:+.2f}%")
print(f"  rho_hdbscan (FZ):   {ganador['rho_hdbscan']:+.4f}")
print(f"  rho_gics    (FZ):   {ganador['rho_gics']:+.4f}")

DIVERGENCIA geométrico vs económico:
  Líder de Fase 1 cae a la posición económica #3.
  Ganador económico ocupaba la posición #2 en Fase 1.

MODELO ÓPTIMO: tfidf_mcs150_ms30_eps0.0_eom
  Representación:     TFIDF
  Reducción vs GICS:  -1.46%
  rho_hdbscan (FZ):   +0.5069
  rho_gics    (FZ):   +0.4859


Guardamos el ranking definitivo con todas las métricas en un archivo CSV para utilizarlo en el cierre del estudio.

In [20]:
df_ranking.to_csv(PATHS["clusters_out"] / "Resumen_Seleccion_Final.csv", index=False)
print("Guardado:", PATHS["clusters_out"] / "Resumen_Seleccion_Final.csv")

Guardado: C:\Users\diego\TFG\outputs\Resumen_Seleccion_Final.csv


## 6. Significación de la diferencia frente a GICS (bootstrap)

Las diferencias en RMSE entre la clasificación HDBSCAN y GICS pueden deberse a una mejora real o a la composición particular del universo de empresas evaluado. Para construir un intervalo de confianza robusto utilizamos un bootstrap por bloques de **ticker**: cada réplica remuestrea empresas con reemplazo y reagrega todas sus regresiones, preservando la dependencia interna de cada empresa y respetando la unidad natural del panel.

Reportamos el intervalo con corrección **BCa** (*bias-corrected & accelerated*) en lugar del percentil simple. El BCa ajusta dos efectos que el percentil ignora: la correción de sesgo $z_0$ (proporción de réplicas por debajo del estimador puntual) y la aceleración $a$ (asimetría de la distribución, estimada por *jackknife*). Es el intervalo de bootstrap estandarizado desde Efron (1987) y resulta especialmente recomendable cuando la distribución bootstrap no es perfectamente simétrica en torno al estimador.

Aplicamos el remuestro únicamente a la reducción de RMSE. La $\Delta\rho$ de Spearman se mantiene como criterio descriptivo de desempate, sin intervalo de confianza, dado que la unidad natural de ese estadístico es el par de empresas dentro de cada clúster y no el ticker individual, lo que rompe la analogía con el bootstrap por bloques de ticker que sí justifica el de RMSE.


In [21]:
# Recolectamos el detalle por regresion (ticker, SSR, n) de cada candidato,
# recorriendo el mismo walk-forward que evaluar_candidato_indice_unico, pero
# guardando el desglose por ticker para HDBSCAN y para GICS.
def recolectar_detalle_candidato(finalista):
    repr_name = finalista["representation"]
    params = finalista["params"]
    matriz_global = matriz_cache[repr_name]
    det_hdb, det_gics = [], []   # listas de (ticker, ssr, n)

    for year_T in anios_T:
        year_eval = year_T + 1
        anios_str = df_nlp["quarter"].astype(str).str[:4]
        idx_train = np.flatnonzero((anios_str.astype(int) <= year_T).to_numpy())
        if idx_train.size < params["min_cluster_size"]:
            continue
        df_train = df_nlp.iloc[idx_train].reset_index(drop=True)
        mat_train = vectores_anio(df_train, repr_name, matriz_global, idx_train)
        _, labels_train = proyectar_y_clusterizar(mat_train, params)
        df_train["label"] = labels_train
        df_T = df_train[df_train["quarter"].astype(str).str.startswith(str(year_T))].copy()
        if df_T.empty:
            continue

        for fecha_ini, fecha_fin in buckets_trimestrales(year_eval):
            tk_label = ticker_label_map(df_T, df_T["label"].values, fecha_corte=fecha_ini)
            if not tk_label:
                continue
            target_quarter = f"{year_eval}Q{((fecha_ini.month - 1) // 3) + 1}"
            gics_eval_map = (df_gics_map[df_gics_map["quarter"] == target_quarter]
                             .set_index("ticker")["gics_sector"])
            universo = [t for t in tk_label.keys()
                        if t in gics_eval_map.index and pd.notna(gics_eval_map[t])]
            if len(universo) < 2:
                continue
            ret_q = retornos_trimestre(df_retornos, fecha_ini, fecha_fin, universo)
            if ret_q is None:
                continue
            universo = [t for t in universo if t in ret_q.columns]
            if len(universo) < 2:
                continue
            tk_label_int = {t: tk_label[t] for t in universo}
            tk_gics_int  = {t: gics_eval_map[t] for t in universo}
            det_hdb  += rmse_indice_unico_detalle(tk_label_int, ret_q)
            det_gics += rmse_indice_unico_detalle(tk_gics_int,  ret_q)
    return det_hdb, det_gics

detalle_por_candidato = {fin["nombre_corto"]: recolectar_detalle_candidato(fin)
                         for fin in finalistas}

# Comprobacion de coherencia: el RMSE reconstruido debe coincidir con el del ranking.
for nombre, (dh, dg) in detalle_por_candidato.items():
    if dh and dg:
        rmse_h = np.sqrt(sum(s for _, s, _ in dh) / sum(n for _, _, n in dh))
        rmse_g = np.sqrt(sum(s for _, s, _ in dg) / sum(n for _, _, n in dg))
        print(f"{nombre:<28} RMSE_hdb={rmse_h:.4f}  RMSE_gics={rmse_g:.4f}  "
              f"red={(rmse_h-rmse_g)/rmse_g*100:+.2f}%")


finbert_mcs400_ms15_eps0.0_leaf RMSE_hdb=0.0313  RMSE_gics=0.0290  red=+8.16%
tfidf_mcs150_ms30_eps0.0_eom RMSE_hdb=0.0286  RMSE_gics=0.0290  red=-1.46%
sbert_mcs150_ms30_eps0.0_leaf RMSE_hdb=0.0277  RMSE_gics=0.0277  red=+0.06%


In [22]:
from scipy.stats import norm

def bootstrap_bca_reduccion_por_ticker(det_hdb, det_gics, B=2000, seed=42):
    """Bootstrap por bloques de ticker sobre la reduccion relativa de RMSE (%).
    Devuelve IC al 95% con correccion BCa (bias-corrected & accelerated).

    El bootstrap percentil simple es valido pero puede sufrir sesgo y asimetria
    cuando la distribucion bootstrap no es centrada en el estimador puntual.
    BCa corrige ambos efectos via dos parametros: la correccion de sesgo z0
    (proporcion de replicas por debajo del estimador) y la aceleracion a
    (asimetria estimada por jackknife). Estandar en Efron (1987).
    """
    rng = np.random.default_rng(seed)

    def por_ticker(detalle):
        d = {}
        for tk, ssr, n in detalle:
            s, nn = d.get(tk, (0.0, 0))
            d[tk] = (s + ssr, nn + n)
        return d
    h = por_ticker(det_hdb)
    g = por_ticker(det_gics)
    tickers = sorted(set(h) | set(g))
    if len(tickers) < 2:
        return None

    h_ssr = np.array([h.get(t, (0.0, 0))[0] for t in tickers])
    h_n   = np.array([h.get(t, (0.0, 0))[1] for t in tickers])
    g_ssr = np.array([g.get(t, (0.0, 0))[0] for t in tickers])
    g_n   = np.array([g.get(t, (0.0, 0))[1] for t in tickers])
    K = len(tickers)

    def reduccion(hs, hn, gs, gn):
        rmse_h = np.sqrt(hs.sum() / max(hn.sum(), 1))
        rmse_g = np.sqrt(gs.sum() / max(gn.sum(), 1))
        return (rmse_h - rmse_g) / rmse_g * 100

    # Estimador puntual sobre la muestra real
    theta_hat = reduccion(h_ssr, h_n, g_ssr, g_n)

    # B replicas: remuestreo de tickers con reemplazo
    reps = np.empty(B)
    for b in range(B):
        idx = rng.integers(0, K, K)
        reps[b] = reduccion(h_ssr[idx], h_n[idx], g_ssr[idx], g_n[idx])

    # Correccion de sesgo z0: proporcion de replicas < estimador puntual.
    prop = np.clip(np.mean(reps < theta_hat), 1e-6, 1 - 1e-6)
    z0 = norm.ppf(prop)

    # Aceleracion a: asimetria estimada por jackknife (dejar fuera un ticker).
    theta_jack = np.empty(K)
    full = np.arange(K)
    for i in range(K):
        m = full != i
        theta_jack[i] = reduccion(h_ssr[m], h_n[m], g_ssr[m], g_n[m])
    theta_bar = theta_jack.mean()
    num = np.sum((theta_bar - theta_jack) ** 3)
    den = 6.0 * (np.sum((theta_bar - theta_jack) ** 2) ** 1.5)
    a = num / den if den != 0 else 0.0

    # Endpoints BCa ajustados.
    def endpoint(alpha):
        z_a = norm.ppf(alpha)
        adj = z0 + (z0 + z_a) / (1 - a * (z0 + z_a))
        return float(np.percentile(reps, 100 * norm.cdf(adj)))

    lo, hi = endpoint(0.025), endpoint(0.975)
    return {
        "theta_hat":            float(theta_hat),
        "reduccion_mediana_pct": round(float(np.percentile(reps, 50)), 3),
        "ic95_bca_inf":         round(lo, 3),
        "ic95_bca_sup":         round(hi, 3),
        "z0":                    round(float(z0), 4),
        "a":                     round(float(a), 6),
        "ic_incluye_0":          bool(lo <= 0 <= hi),
        "p(red>=0)":             round(float(np.mean(reps >= 0)), 3),
        "replicas":              reps,
    }


filas_boot, replicas_rmse = [], {}
for fin in finalistas:
    dh, dg = detalle_por_candidato[fin["nombre_corto"]]
    res = bootstrap_bca_reduccion_por_ticker(dh, dg, B=2000, seed=42)
    if res is None:
        continue
    replicas_rmse[fin["nombre_corto"]] = res.pop("replicas")
    filas_boot.append({
        "nombre_corto":   fin["nombre_corto"],
        "representation": fin["representation"],
        **res,
    })

df_boot = pd.DataFrame(filas_boot)
df_boot.to_csv(PATHS["clusters_out"] / "Bootstrap_Reduccion_RMSE.csv", index=False)
# Replicas crudas para la Figura 6b (violines) en NB06.
pd.DataFrame(replicas_rmse).to_csv(
    PATHS["clusters_out"] / "Bootstrap_Replicas_RMSE.csv", index=False
)

print("Bootstrap por bloques de ticker (B=2000, IC BCa 95%):\n")
for _, r in df_boot.iterrows():
    signif = "no distinguible de 0" if r["ic_incluye_0"] else "distinguible de 0"
    print(f"  {r['nombre_corto']:<32} mediana={r['reduccion_mediana_pct']:+.2f}%  "
          f"IC95_BCa=[{r['ic95_bca_inf']:+.2f}, {r['ic95_bca_sup']:+.2f}]  -> {signif}")
df_boot


Bootstrap por bloques de ticker (B=2000, IC BCa 95%):

  finbert_mcs400_ms15_eps0.0_leaf  mediana=+8.12%  IC95_BCa=[+6.80, +9.94]  -> distinguible de 0
  tfidf_mcs150_ms30_eps0.0_eom     mediana=-1.44%  IC95_BCa=[-2.37, -0.65]  -> distinguible de 0
  sbert_mcs150_ms30_eps0.0_leaf    mediana=+0.04%  IC95_BCa=[-0.57, +0.78]  -> no distinguible de 0


,nombre_corto,representation,theta_hat,reduccion_mediana_pct,ic95_bca_inf,ic95_bca_sup,z0,a,ic_incluye_0,p(red>=0)
0,finbert_mcs400_ms15_eps0.0_leaf,finbert,8.159775,8.121,6.796,9.936,0.0389,0.011192,False,1.000
1,tfidf_mcs150_ms30_eps0.0_eom,tfidf,-1.456583,-1.441,-2.367,-0.647,-0.0426,-0.008520,False,0.000
2,sbert_mcs150_ms30_eps0.0_leaf,sbert,0.058782,0.039,-0.572,0.782,0.0552,0.009331,True,0.543


## 7. Dispersión interanual de la reducción de RMSE

La métrica del ranking colapsa los cuatro años de evaluación en un único RMSE agregado. Como complemento, recuperamos el desglose anual ya guardado en `Resultados_IndiceUnico_WF_anual.csv` y mostramos la reducción frente a GICS año a año. Enseñar la dispersión interanual evita leer un único punto como si fuera estable: una ventaja media puede deberse a un solo año favorable.

In [23]:
ruta_anual = PATHS["clusters_out"] / "Resultados_IndiceUnico_WF_anual.csv"
df_anual_iu = pd.read_csv(ruta_anual)
df_anual_iu["reduccion_pct"] = (
    (df_anual_iu["rmse_hdbscan"] - df_anual_iu["rmse_gics"])
    / df_anual_iu["rmse_gics"] * 100
)

# Tabla pivote: filas = candidato, columnas = año evaluado, valor = reduccion %.
tabla_interanual = (
    df_anual_iu.pivot_table(index="nombre_corto", columns="year_eval",
                            values="reduccion_pct")
    .round(2)
)
tabla_interanual["media"] = tabla_interanual.mean(axis=1).round(2)
tabla_interanual["std"]   = df_anual_iu.groupby("nombre_corto")["reduccion_pct"].std().round(2)
print("Reducción de RMSE vs GICS por año evaluado (%):")
tabla_interanual


Reducción de RMSE vs GICS por año evaluado (%):


year_eval,2021,2022,2023,2024,media,std
nombre_corto,,,,,,
finbert_mcs400_ms15_eps0.0_leaf,10.70,8.51,5.13,1.42,6.44,4.06
sbert_mcs150_ms30_eps0.0_leaf,-0.00,0.45,-0.36,0.22,0.08,0.34
tfidf_mcs150_ms30_eps0.0_eom,0.16,-3.83,-0.64,0.56,-0.94,1.99


## 8. Persistencia de artefactos para visualizaciones

Generamos los CSV que el cuaderno 06 (visualizaciones) consume de tal manera que en el cuaderno de visualizaciones no se tenga que ejecutar ninguna tarea analítica de peso.


In [24]:
# PERSISTENCIA DE ARTEFACTOS PARA VISUALIZACIONES (NB06)
# Genera los CSV que el siguiente cuaderno lee para dibujar las Figuras.

VENTANA_VIZ = max(anios_T)  # T=2023: ventana de referencia para figuras de composicion


# P1 — Composicion de clusteres
def composicion_y_ruido_finalista(finalista):
    repr_name = finalista["representation"]
    params = finalista["params"]
    matriz_global = matriz_cache[repr_name]
    filas, filas_ruido = [], []

    company_map = (
        df_nlp.drop_duplicates("ticker").set_index("ticker")["company"]
        if "company" in df_nlp.columns else None
    )

    for year_T in anios_T:
        anios_str = df_nlp["quarter"].astype(str).str[:4]
        idx_train = np.flatnonzero((anios_str.astype(int) <= year_T).to_numpy())
        if idx_train.size < params["min_cluster_size"]:
            continue

        df_train = df_nlp.iloc[idx_train].reset_index(drop=True)
        mat_train = vectores_anio(df_train, repr_name, matriz_global, idx_train)
        _, labels_train = proyectar_y_clusterizar(mat_train, params)
        df_train["label"] = labels_train

        df_T = df_train[df_train["quarter"].astype(str).str.startswith(str(year_T))].copy()
        if df_T.empty:
            continue

        # Una fila por ticker (su ultima earnings call del ano T), incluyendo -1
        # para contar el ruido (ticker_label_map estandar lo excluiria).
        tmp = (df_T[["ticker", "quarter", "label"]]
               .sort_values("quarter").groupby("ticker").tail(1)
               .set_index("ticker")["label"].to_dict())

        gics_map_T = (
            df_gics_map[df_gics_map["quarter"] == f"{year_T}Q4"]
            .set_index("ticker")["gics_sector"]
        )

        n_ruido = sum(1 for v in tmp.values() if v == -1)
        n_total = len(tmp)
        filas_ruido.append({
            "representation":  repr_name,
            "experiment_id":   finalista["nombre_corto"],
            "year_T":          year_T,
            "n_tickers":       n_total,
            "n_ruido":         n_ruido,
            "n_clasificados":  n_total - n_ruido,
            "tasa_ruido":      round(n_ruido / n_total, 4) if n_total else np.nan,
        })

        for ticker, label in tmp.items():
            if label == -1:
                continue
            filas.append({
                "representation": repr_name,
                "experiment_id":  finalista["nombre_corto"],
                "year_T":         year_T,
                "ticker":         ticker,
                "company":        (company_map.get(ticker, "") if company_map is not None else ""),
                "cluster_label":  int(label),
                "gics_sector":    gics_map_T.get(ticker, np.nan),
            })

    return pd.DataFrame(filas), pd.DataFrame(filas_ruido)


comp_list, ruido_list = [], []
for fin in finalistas:
    c, r = composicion_y_ruido_finalista(fin)
    if not c.empty:
        comp_list.append(c)
    if not r.empty:
        ruido_list.append(r)

df_composicion = pd.concat(comp_list, ignore_index=True)
df_ruido = pd.concat(ruido_list, ignore_index=True)
df_composicion.to_csv(PATHS["clusters_out"] / "Composicion_Clusteres_WF.csv", index=False)
df_ruido.to_csv(PATHS["clusters_out"] / "Ruido_Por_Ventana.csv", index=False)
print("P1 -> Composicion_Clusteres_WF.csv | Ruido_Por_Ventana.csv")
print(df_ruido.to_string(index=False))

# P2 — Tamanos de cluster por modelo y ventana
df_tamanos = (
    df_composicion.groupby(["representation", "year_T", "cluster_label"])
    .size().rename("n_empresas").reset_index()
    .sort_values(["representation", "year_T", "n_empresas"],
                 ascending=[True, True, False])
)
df_tamanos.to_csv(PATHS["clusters_out"] / "Tamanos_Clusteres_WF.csv", index=False)
print("\nP2 -> Tamanos_Clusteres_WF.csv")
print(df_tamanos[df_tamanos["year_T"] == VENTANA_VIZ].to_string(index=False))


# P3 — Modelos supervivientes en cada etapa del proceso de selección
df_reg = pd.read_csv(PATHS["finalistas_fase1"].parent / "registry.csv")
filas_embudo = []
for repr_name in ["tfidf", "finbert", "sbert"]:
    sub = df_reg[df_reg["representation"] == repr_name].copy()
    n0 = len(sub)
    # Etapa 1: dbcv/sil definidos, ruido<0.5, validos en las 3 ventanas
    e1 = sub[
        sub["dbcv"].notna() & sub["silhouette"].notna()
        & (sub["noise_ratio"] < 0.50)
        & (sub.get("dbcv_nfolds", 3) == 3)
        & (sub.get("silhouette_nfolds", 3) == 3)
    ]
    n1 = len(e1)
    # Etapa 2: dbcv y sil >= medianas de los supervivientes de etapa 1
    med_dbcv, med_sil = e1["dbcv"].median(), e1["silhouette"].median()
    e2 = e1[(e1["dbcv"] >= med_dbcv) & (e1["silhouette"] >= med_sil)]
    n2 = len(e2)
    filas_embudo.append({
        "representation":            repr_name,
        "etapa_0_total":             n0,
        "etapa_1_estabilidad":       n1,
        "etapa_2_calidad_relativa":  n2,
        "etapa_3_finalista":         1,
    })
df_embudo = pd.DataFrame(filas_embudo)
df_embudo.to_csv(PATHS["clusters_out"] / "Embudo_Filtro.csv", index=False)
print("\nP3 -> Embudo_Filtro.csv")
print(df_embudo.to_string(index=False))


# P4 — Datos del cuadrante geometria vs economia
df_fin_csv = pd.read_csv(PATHS["finalistas_fase1"])
df_iu = pd.read_csv(PATHS["clusters_out"] / "Resultados_IndiceUnico_WF.csv")
df_cuad = df_fin_csv[["experiment_id", "representation", "dbcv", "silhouette",
                      "noise_ratio", "n_clusters"]].merge(
    df_iu[["nombre_corto", "reduccion_pct", "rmse_hdbscan", "rmse_gics"]],
    left_on="experiment_id", right_on="nombre_corto", how="left"
).drop(columns="nombre_corto")
df_cuad.to_csv(PATHS["clusters_out"] / "Cuadrante_Geom_vs_Econ.csv", index=False)
print("\nP4 -> Cuadrante_Geom_vs_Econ.csv")
print(df_cuad.to_string(index=False))


P1 -> Composicion_Clusteres_WF.csv | Ruido_Por_Ventana.csv
representation                   experiment_id  year_T  n_tickers  n_ruido  n_clasificados  tasa_ruido
       finbert finbert_mcs400_ms15_eps0.0_leaf    2020        451       17             434      0.0377
       finbert finbert_mcs400_ms15_eps0.0_leaf    2021        450      217             233      0.4822
       finbert finbert_mcs400_ms15_eps0.0_leaf    2022        449      228             221      0.5078
       finbert finbert_mcs400_ms15_eps0.0_leaf    2023        454      181             273      0.3987
         tfidf    tfidf_mcs150_ms30_eps0.0_eom    2020        451      107             344      0.2373
         tfidf    tfidf_mcs150_ms30_eps0.0_eom    2021        450      133             317      0.2956
         tfidf    tfidf_mcs150_ms30_eps0.0_eom    2022        449      157             292      0.3497
         tfidf    tfidf_mcs150_ms30_eps0.0_eom    2023        454      195             259      0.4295
         sbert

In [25]:
#   Genera dos CSV especificos del finalista TF-IDF en T=VENTANA_VIZ (2023):
#     1) Coordenadas UMAP 2D para la figura "mapa del mercado" (NB06)
#     2) Top terminos TF-IDF por cluster para la figura "perfil lexico"

from sklearn.feature_extraction.text import TfidfVectorizer

# --- Localizar el finalista TF-IDF ---
finalista_tfidf = next(f for f in finalistas if f["representation"] == "tfidf")
params_tfidf = finalista_tfidf["params"]

# --- Reconstruir la matriz TF-IDF de la ventana <= VENTANA_VIZ (idem a vectores_anio) ---
anios_str = df_nlp["quarter"].astype(str).str[:4]
idx_train = np.flatnonzero((anios_str.astype(int) <= VENTANA_VIZ).to_numpy())
df_train = df_nlp.iloc[idx_train].reset_index(drop=True)

textos = df_train["presentation_limpia"].astype(str).tolist()
vectorizer_local = TfidfVectorizer(
    max_df=PARAMS_TFIDF["max_df"],
    min_df=PARAMS_TFIDF["min_df"],
    max_features=PARAMS_TFIDF["max_features"],
    ngram_range=(1, 2),
    sublinear_tf=True,
)
matriz_local = vectorizer_local.fit_transform(textos)
vocabulario = np.array(vectorizer_local.get_feature_names_out())

# --- Re-clusterizar para obtener etiquetas (mismas semillas -> resultado deterministico) ---
mat_dense = matriz_local.toarray()
embeddings_50, labels_train = proyectar_y_clusterizar(mat_dense, params_tfidf)
df_train["label"] = labels_train


# A) UMAP 2D ILUSTRATIVO — coordenadas para el "mapa del mercado"
#   Proyeccion *separada* a 2D (random_state=123) sobre la misma matriz
#   TF-IDF, con la misma metrica (coseno) y n_neighbors=15,
#   para que la geometria 2D sea lo mas fiel posible al espacio analitico.

#   IMPORTANTE: las distancias en 2D NO reflejan necesariamente la
#   separacion en R^50 donde se hizo el clustering.

SEED_UMAP_2D = 123  # independiente del 42 analitico (memoria del usuario)

reducer_2d = umap.UMAP(
    n_neighbors=UMAP_PARAMS["n_neighbors"],
    n_components=2,
    metric=UMAP_PARAMS["metric"],
    random_state=SEED_UMAP_2D,
)
coords_2d = reducer_2d.fit_transform(mat_dense)

# Tomamos la última earnings call del ano VENTANA_VIZ por ticker (mismo criterio
# que ticker_label_map). Asi una empresa es un punto en el plano.
df_T = df_train[df_train["quarter"].astype(str).str.startswith(str(VENTANA_VIZ))].copy()
df_T["x"] = coords_2d[df_T.index, 0]
df_T["y"] = coords_2d[df_T.index, 1]
df_T = df_T.sort_values("quarter").groupby("ticker").tail(1)

# Anotar GICS vigente en VENTANA_VIZ Q4
gics_map_T = (df_gics_map[df_gics_map["quarter"] == f"{VENTANA_VIZ}Q4"]
              .set_index("ticker")["gics_sector"])
df_T["gics_sector"] = df_T["ticker"].map(gics_map_T)

# Excluir ruido del CSV (la figura solo dibuja puntos clasificados)
df_umap2d = df_T[df_T["label"] != -1][["ticker", "label", "gics_sector", "x", "y"]].copy()
df_umap2d = df_umap2d.rename(columns={"label": "cluster_label"})
df_umap2d["seed_umap2d"] = SEED_UMAP_2D  # para trazabilidad en el pie de figura
df_umap2d.to_csv(PATHS["clusters_out"] / "UMAP2D_TFIDF.csv", index=False)
print(f"-> UMAP2D_TFIDF.csv  ({len(df_umap2d)} puntos, seed={SEED_UMAP_2D})")


# B) TOP TERMINOS POR CLUSTER TF-IDF — perfil lexico
#   Para los 6 clusteres mas grandes, los 8 terminos con mayor peso TF-IDF
#   medio entre los documentos del cluster. Estos son los terminos que mas
#   contribuyen a la identidad lexica del grupo.

N_CLUSTERES = 6
TOP_K = 8

# Tamano de cada cluster en la ventana VENTANA_VIZ (no en todo el historico,
# para que sea coherente con el resto de figuras de la ventana).
clusters_T = (df_T[df_T["label"] != -1].groupby("label").size()
              .sort_values(ascending=False))
top_clusters = clusters_T.head(N_CLUSTERES).index.tolist()

filas_top = []
for cl in top_clusters:
    # Documentos del cluster en TODO el historico <= VENTANA_VIZ (mejor senal
    # lexica que solo los del ultimo ano, porque el cluster se forma con todo).
    idx_cl = np.flatnonzero(df_train["label"].values == cl)
    if len(idx_cl) == 0:
        continue
    # Peso medio por termino dentro del cluster (matriz dispersa, sumamos y dividimos)
    sub = matriz_local[idx_cl]
    pesos_medios = np.asarray(sub.mean(axis=0)).ravel()
    top_idx = np.argsort(-pesos_medios)[:TOP_K]
    n_doc = len(idx_cl)
    n_emp_T = int(clusters_T.loc[cl])
    for rango, j in enumerate(top_idx, start=1):
        filas_top.append({
            "cluster_label":   int(cl),
            "n_docs_historico": n_doc,
            "n_empresas_T":    n_emp_T,
            "rango":           rango,
            "termino":         vocabulario[j],
            "peso_medio":      float(pesos_medios[j]),
        })

df_top = pd.DataFrame(filas_top)
df_top.to_csv(PATHS["clusters_out"] / "TopTerminos_TFIDF.csv", index=False)
print(f"-> TopTerminos_TFIDF.csv ({N_CLUSTERES} clusteres x {TOP_K} terminos)")
# Vista rapida
for cl in top_clusters:
    terms = df_top[df_top["cluster_label"] == cl].sort_values("rango")
    if terms.empty:
        continue
    n_emp = int(terms["n_empresas_T"].iloc[0])
    palabras = ", ".join(terms["termino"].tolist())
    print(f"  C{cl:>2} (n={n_emp:>3} empresas en T={VENTANA_VIZ}): {palabras}")


-> UMAP2D_TFIDF.csv  (259 puntos, seed=123)
-> TopTerminos_TFIDF.csv (6 clusteres x 8 terminos)
  C13 (n= 41 empresas en T=2023): client, revenue, product, margin, customer, platform, fee, organic
  C10 (n= 24 empresas en T=2023): brand, consumer, sale growth, sale, category, pricing, net sale, restaurant
  C 3 (n= 19 empresas en T=2023): rent, lease, noi, occupancy, ffo, tenant, leasing, property
  C 5 (n= 19 empresas en T=2023): patient, procedure, clinical, diagnostic, organic, therapy, revenue, china
  C 8 (n= 18 empresas en T=2023): utility, electric, rate case, transmission, energy, weather, megawatt, gas
  C 6 (n= 16 empresas en T=2023): patient, disease, study, trial, clinical, treatment, therapy, medicine
